# Music Annotation Project
## Notebook 89R — A5 Candidate Reconstruction and Multi-Threshold Visual Audit

**CPU runtime.** This notebook reads one completed Notebook 88 run. It does not retrain, refit or recalibrate A5.

### Correct interpretation
Notebook 88 uses the established truth labels:

- `0 = ANNOTATION`
- `1 = PRINT`
- `2 = UNCERTAIN`

The probability used here is treated as **PRINT-oriented** only after a mandatory empirical directionality audit. A selected A5 pixel is therefore applied by copying the exact observed byte into the A1 baseline.

Visual colours:

- **green = genuine print restored/preserved**
- **red = annotation wrongly retained**
- **purple = truth uncertain**

The notebook fails closed if the high-probability tail is not PRINT-dominant, if required A1/observed/paper sources cannot be resolved, or if candidate assembly changes any non-selected pixel.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip install -q opencv-python-headless pandas numpy pyarrow matplotlib


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import json, hashlib, math, shutil, sys, platform, zipfile, gc
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path('/content/drive/MyDrive/MusicAnnotationProject')
SOURCE_RUN_OVERRIDE = None
N88_ROOT = ROOT / 'a5_contextual_safe_cleanup_v88'
OUTPUT_ROOT = ROOT / 'notebook_89r_a5_candidate_audit'

THRESHOLDS = [0.995, 0.990, 0.980, 0.950]
MIN_PRINT_RUN = 2
REVIEW_RADIUS = 3
OVERLAY_ALPHA = 0.72
CROP_PADDING = 120
MIN_CROP_SIDE = 520
MAX_CROP_SIDE = 1400
MAX_QUEUE = 50

TRUTH_ANNOTATION = 0
TRUTH_PRINT = 1
TRUTH_UNCERTAIN = 2

RUN = datetime.now().strftime('%Y%m%d_%H%M%S') + '_notebook89r_a5_candidate_audit'
OUT = OUTPUT_ROOT / RUN
TABLES = OUT / 'tables'
CROPS = OUT / 'crops'
WHOLE = OUT / 'whole_pages'
VISUAL = OUT / 'visual_audit'
AUDIT = OUT / 'audit'
UPLOAD = OUT / 'upload'
for p in [OUT, TABLES, CROPS, WHOLE, VISUAL, AUDIT, UPLOAD]:
    p.mkdir(parents=True, exist_ok=False)

def require(condition, message):
    if not condition:
        raise RuntimeError(message)

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for block in iter(lambda: f.read(1048576), b''):
            h.update(block)
    return h.hexdigest()

def load_gray(path):
    path = Path(path)
    if path.suffix.lower() == '.npy':
        image = np.load(path, allow_pickle=False)
        require(image.ndim == 2, f'{path}: expected 2D array, got {image.shape}')
        require(np.isfinite(image).all(), f'{path}: non-finite values')
        if image.dtype == np.bool_:
            return image.astype(np.uint8) * 255
        require(image.min() >= 0 and image.max() <= 255, f'{path}: values outside [0,255]')
        return image.astype(np.uint8)
    image = cv2.imread(str(path), cv2.IMREAD_GRAYSCALE)
    require(image is not None, f'Could not read image: {path}')
    return image

print('Output:', OUT)


## 1. Resolve Notebook 88 predictions and enforce semantics


In [ ]:
PRED_NAMES = [
    'a5_primary_oof_predictions.parquet',
    'all_eligible_oof_predictions.parquet',
    'a5_all_eligible_oof_predictions.parquet',
    'oof_predictions.parquet',
]

def completed_runs(root):
    hits = []
    if root.exists():
        for name in PRED_NAMES:
            for f in root.rglob(name):
                for p in [f.parent, f.parent.parent]:
                    if p.is_dir() and any((p/'tables'/n).exists() or (p/n).exists() for n in PRED_NAMES):
                        hits.append(p)
    unique = {str(p): p for p in hits}
    return sorted(unique.values(), key=lambda p: p.stat().st_mtime)

SOURCE = Path(SOURCE_RUN_OVERRIDE) if SOURCE_RUN_OVERRIDE else completed_runs(N88_ROOT)[-1]

def find_file(names):
    for name in names:
        for path in [SOURCE/'tables'/name, SOURCE/name]:
            if path.exists():
                return path
    return None

PRED = find_file(PRED_NAMES)
require(PRED is not None, 'Notebook 88 prediction parquet not found.')
full = pd.read_parquet(PRED)

aliases = {
    'page': ['page', 'page_id', 'stem'],
    'source_group': ['source_group', 'group'],
    'y': ['y', 'row', 'pixel_y'],
    'x': ['x', 'col', 'pixel_x'],
    # Deliberately prefer explicit PRINT probability fields.
    'prob': ['p_print_a5', 'p_print', 'print_probability', 'calibrated_print_probability',
             'p_annotation_a5', 'p_clean', 'clean_probability', 'prob_clean'],
    'truth': ['truth', 'label', 'truth_class'],
}
resolved = {k: next((name for name in names if name in full.columns), None) for k, names in aliases.items()}
require(all(resolved[k] for k in ['page','y','x','truth','prob']),
        f'Could not resolve prediction schema. Columns: {list(full.columns)}')
full = full.rename(columns={v:k for k,v in resolved.items() if v and v != k}).copy()
if 'source_group' not in full:
    full['source_group'] = 'UNKNOWN'
for c in ['y','x','truth']:
    full[c] = pd.to_numeric(full[c], errors='raise').astype(int)
full['prob'] = pd.to_numeric(full.prob, errors='raise').astype(np.float64)
full['page'] = full.page.astype(str)
full['source_group'] = full.source_group.astype(str)
require(np.isfinite(full.prob).all(), 'Non-finite probabilities found.')
require(not full.duplicated(['page','y','x']).any(), 'Duplicate page/y/x rows found.')
require(set(full.truth.unique()).issubset({0,1,2}),
        'Expected truth labels 0=ANNOTATION, 1=PRINT, 2=UNCERTAIN.')

# Mandatory directionality audit. The high tail must behave as P(PRINT).
direction_rows = []
for threshold in THRESHOLDS:
    selected = full.prob.to_numpy() >= threshold
    counts = full.loc[selected, 'truth'].value_counts().reindex([0,1,2], fill_value=0)
    require(int(counts.sum()) == int(selected.sum()), f'{threshold}: class counts do not sum to selected total.')
    direction_rows.append({
        'threshold': threshold,
        'selected': int(selected.sum()),
        'annotation_truth': int(counts.loc[0]),
        'print_truth': int(counts.loc[1]),
        'uncertain_truth': int(counts.loc[2]),
        'print_fraction_scored': float(counts.loc[1] / max(counts.loc[0] + counts.loc[1], 1)),
    })
direction = pd.DataFrame(direction_rows)
direction.to_csv(AUDIT/'probability_directionality.csv', index=False)
require(direction.iloc[0].print_fraction_scored > 0.50,
        'The >=0.995 tail is not PRINT-dominant. Refusing to assemble a PRINT-restoration candidate.')

(AUDIT/'schema_resolution.json').write_text(json.dumps({
    'source': str(SOURCE), 'predictions': str(PRED), 'resolved': resolved,
    'truth_semantics': {'0':'ANNOTATION','1':'PRINT','2':'UNCERTAIN'},
    'probability_semantics': 'empirically verified PRINT-oriented',
    'rows': len(full),
}, indent=2))
print('Source:', SOURCE)
print(direction.to_string(index=False))


## 2. Resolve the exact Notebook 88 page arrays


In [ ]:
SUP = {'.png','.jpg','.jpeg','.tif','.tiff','.bmp'}
def imap(folder):
    folder = Path(folder)
    return {p.stem:p for p in folder.iterdir() if p.is_file() and p.suffix.lower() in SUP} if folder.exists() else {}

obsmap = imap(ROOT/'aligned')

CHANNEL_CACHE_CANDIDATES = [
    SOURCE/'observable_channel_cache', SOURCE/'channel_cache',
    SOURCE/'cache'/'observable_channels', SOURCE/'cache'
]
CHANNEL_CACHE = next((p for p in CHANNEL_CACHE_CANDIDATES if p.exists() and p.is_dir()), None)
require(CHANNEL_CACHE is not None, 'Could not locate Notebook 88 observable-channel cache.')

def nested(page, names):
    for root in [SOURCE/'whole_pages'/page, SOURCE/'pages'/page, SOURCE/page]:
        for name in names:
            path = root/name
            if path.exists():
                return path
    return None

def cached_channel(page, names):
    for root in [CHANNEL_CACHE/page, CHANNEL_CACHE]:
        for name in names:
            for path in [root/f'{name}.npy', root/f'{page}_{name}.npy']:
                if path.exists():
                    return path
    return None

files = {}
missing = []
for page in sorted(full.page.unique()):
    observed = cached_channel(page, ['observed','obs']) or nested(page, ['observed.png','observed_page.png']) or obsmap.get(page)
    a1 = cached_channel(page, ['a1','a1_source','a1_fallback','baseline','corrected']) or nested(page, ['a1.png','a1_fallback.png','baseline.png','v7_8.png'])
    paper = cached_channel(page, ['paper','paper_estimate','background','background_surface'])
    if paper is None and a1 is not None:
        sibling = Path(a1).parent/'paper.npy'
        paper = sibling if sibling.exists() else None
    if observed is None or a1 is None or paper is None:
        missing.append({'page':page,'observed':str(observed) if observed else None,
                        'a1':str(a1) if a1 else None,'paper':str(paper) if paper else None})
    else:
        files[page] = {'observed':Path(observed),'a1':Path(a1),'paper':Path(paper)}

pd.DataFrame(missing).to_csv(AUDIT/'missing_arrays.csv', index=False)
require(not missing, f'{len(missing)} pages lack observed, A1 or paper arrays. See missing_arrays.csv.')
pd.DataFrame([{'page':p, **{k:str(v) for k,v in q.items()}} for p,q in files.items()]).to_csv(AUDIT/'array_resolution.csv', index=False)
print('Resolved page arrays:', len(files))


## 3. Recreate threshold selections and assemble A5 candidates


In [ ]:
def run_filter(frame, selected):
    final = np.asarray(selected, bool).copy()
    for page, g in frame.groupby('page', sort=False):
        idx = g.index.to_numpy()
        image = load_gray(files[page]['observed'])
        mask = np.zeros(image.shape, np.uint8)
        take = idx[final[idx]]
        if not len(take):
            continue
        mask[frame.loc[take,'y'].to_numpy(), frame.loc[take,'x'].to_numpy()] = 1
        _, labels, stats, _ = cv2.connectedComponentsWithStats(mask, 8)
        ids = labels[frame.loc[take,'y'].to_numpy(), frame.loc[take,'x'].to_numpy()]
        final[take[stats[ids, cv2.CC_STAT_AREA] < MIN_PRINT_RUN]] = False
    return final

for threshold in THRESHOLDS:
    full[f'sel_{threshold:.3f}'] = run_filter(full, full.prob.to_numpy() >= threshold)

# Stable connected eligible regions for crop identity.
compmaps, statsmap = {}, {}
for page, g in full.groupby('page'):
    image = load_gray(files[page]['observed'])
    eligible = np.zeros(image.shape, np.uint8)
    eligible[g.y.to_numpy(), g.x.to_numpy()] = 1
    _, labels, stats, _ = cv2.connectedComponentsWithStats(eligible, 8)
    compmaps[page], statsmap[page] = labels, stats

summary_rows, page_rows, component_rows = [], [], []
for threshold in THRESHOLDS:
    col = f'sel_{threshold:.3f}'
    selected = full[col]
    summary_rows.append({
        'threshold':threshold,
        'selected':int(selected.sum()),
        'print_restored':int((selected & (full.truth==TRUTH_PRINT)).sum()),
        'annotation_retained':int((selected & (full.truth==TRUTH_ANNOTATION)).sum()),
        'truth_uncertain':int((selected & (full.truth==TRUTH_UNCERTAIN)).sum()),
        'coverage':float(selected.mean()),
    })
    for page, g in full.groupby('page'):
        q = g[g[col]].copy()
        tp = int((q.truth==TRUTH_PRINT).sum())
        fp = int((q.truth==TRUTH_ANNOTATION).sum())
        unc = int((q.truth==TRUTH_UNCERTAIN).sum())
        page_rows.append({'threshold':threshold,'page':page,'source_group':g.source_group.iloc[0],
                          'selected':len(q),'print_restored':tp,'annotation_retained':fp,
                          'truth_uncertain':unc,'utility':tp-5*fp})
        if q.empty:
            continue
        q['component_id'] = compmaps[page][q.y.to_numpy(), q.x.to_numpy()]
        for cid, z in q[q.component_id>0].groupby('component_id'):
            tp = int((z.truth==TRUTH_PRINT).sum())
            fp = int((z.truth==TRUTH_ANNOTATION).sum())
            unc = int((z.truth==TRUTH_UNCERTAIN).sum())
            catastrophic = fp >= 50 or (fp/max(tp,1) > 1 and fp >= 20)
            st = statsmap[page][int(cid)]
            component_rows.append({'threshold':threshold,'page':page,'source_group':z.source_group.iloc[0],
                'component_id':int(cid),'selected':len(z),'print_restored':tp,'annotation_retained':fp,
                'truth_uncertain':unc,'catastrophic':catastrophic,
                'utility':tp-5*fp-(500 if catastrophic else 0),
                'x':int(st[0]),'y':int(st[1]),'w':int(st[2]),'h':int(st[3])})

summary = pd.DataFrame(summary_rows)
summary['annotation_retained_per_print_restored'] = summary.annotation_retained / np.maximum(summary.print_restored, 1)
pages = pd.DataFrame(page_rows)
components = pd.DataFrame(component_rows)
summary.to_csv(TABLES/'threshold_summary.csv', index=False)
pages.to_csv(TABLES/'results_by_page.csv', index=False)
components.to_csv(TABLES/'results_by_component.csv', index=False)
full.to_parquet(TABLES/'threshold_decisions.parquet', index=False)
display(summary)

# Assemble and verify every whole-page candidate.
for page, g in full.groupby('page', sort=False):
    observed = load_gray(files[page]['observed'])
    a1 = load_gray(files[page]['a1'])
    paper = load_gray(files[page]['paper'])
    require(observed.shape == a1.shape == paper.shape, f'{page}: observed/A1/paper shape mismatch.')
    d = WHOLE/page
    d.mkdir(parents=True, exist_ok=False)
    cv2.imwrite(str(d/'observed.png'), observed)
    cv2.imwrite(str(d/'a1.png'), a1)
    for threshold in THRESHOLDS:
        selected = g[f'sel_{threshold:.3f}'].to_numpy(bool)
        ys = g.y.to_numpy(int); xs = g.x.to_numpy(int)
        candidate = a1.copy()
        candidate[ys[selected], xs[selected]] = observed[ys[selected], xs[selected]]
        selected_map = np.zeros(observed.shape, bool)
        selected_map[ys[selected], xs[selected]] = True
        require(np.array_equal(candidate[~selected_map], a1[~selected_map]),
                f'{page} {threshold}: non-selected pixels changed.')
        require(np.array_equal(candidate[selected_map], observed[selected_map]),
                f'{page} {threshold}: selected pixels do not equal observed bytes.')
        cv2.imwrite(str(d/f'a5_{threshold:.3f}.png'), candidate)
        difference = np.zeros_like(observed); difference[candidate != a1] = 255
        cv2.imwrite(str(d/f'difference_from_a1_{threshold:.3f}.png'), difference)
print('Whole-page A5 candidates written:', full.page.nunique())


## 4. High-resolution truth overlays and component review sheets


In [ ]:
GREEN = (35,190,70)      # true PRINT restored
RED = (20,20,240)        # ANNOTATION wrongly retained
PURPLE = (210,40,210)    # truth uncertain

def dilate(mask, radius=REVIEW_RADIUS):
    if radius <= 0:
        return mask.copy()
    kernel = np.ones((2*radius+1,2*radius+1), np.uint8)
    return cv2.dilate(mask.astype(np.uint8), kernel) > 0

def overlay(gray, layers):
    base = cv2.cvtColor(gray, cv2.COLOR_GRAY2BGR)
    colour = base.copy(); active = np.zeros(gray.shape, bool)
    for mask, bgr in layers:
        colour[mask] = bgr; active |= mask
    if active.any():
        mixed = cv2.addWeighted(base, 1-OVERLAY_ALPHA, colour, OVERLAY_ALPHA, 0)
        base[active] = mixed[active]
    return base

def cropbox(shape,x,y,w,h):
    H,W = shape
    side = min(MAX_CROP_SIDE, max(MIN_CROP_SIDE, w+2*CROP_PADDING, h+2*CROP_PADDING))
    cx,cy = x+w//2,y+h//2
    x0=max(0,min(max(W-side,0),cx-side//2)); y0=max(0,min(max(H-side,0),cy-side//2))
    return x0,y0,min(W,x0+side),min(H,y0+side)

def letterbox(image, width=520, height=520, header=42):
    if image.ndim==2: image=cv2.cvtColor(image,cv2.COLOR_GRAY2BGR)
    avail=height-header; h,w=image.shape[:2]; scale=min(width/w,avail/h)
    resized=cv2.resize(image,(max(1,round(w*scale)),max(1,round(h*scale))),interpolation=cv2.INTER_AREA if scale<1 else cv2.INTER_NEAREST)
    panel=np.full((height,width,3),255,np.uint8); y=header+(avail-resized.shape[0])//2; x=(width-resized.shape[1])//2
    panel[y:y+resized.shape[0],x:x+resized.shape[1]]=resized
    return panel

def save_component_sheet(images,titles,path,title,columns=4):
    pw,ph,th=520,520,76; rows=math.ceil(len(images)/columns)
    sheet=np.full((th+rows*ph,columns*pw,3),255,np.uint8)
    cv2.putText(sheet,title[:220],(14,34),cv2.FONT_HERSHEY_SIMPLEX,.62,(25,25,25),1,cv2.LINE_AA)
    cv2.putText(sheet,'Green = print restored; red = annotation retained; purple = truth uncertain',(14,61),cv2.FONT_HERSHEY_SIMPLEX,.53,(50,50,50),1,cv2.LINE_AA)
    for i,(image,panel_title) in enumerate(zip(images,titles)):
        panel=letterbox(image); cv2.putText(panel,str(panel_title)[:58],(10,27),cv2.FONT_HERSHEY_SIMPLEX,.55,(30,30,30),1,cv2.LINE_AA)
        r,c=divmod(i,columns); sheet[th+r*ph:th+(r+1)*ph,c*pw:(c+1)*pw]=panel
    path=Path(path).with_suffix('.jpg'); path.parent.mkdir(parents=True,exist_ok=True)
    require(cv2.imwrite(str(path),sheet,[cv2.IMWRITE_JPEG_QUALITY,94]),f'Could not write {path}')
    return path

rows=[]
union=(components.groupby(['page','source_group','component_id'],as_index=False)
       .agg(max_print_restored=('print_restored','max'),max_annotation_retained=('annotation_retained','max'),
            max_uncertain=('truth_uncertain','max'),ever_catastrophic=('catastrophic','max'),
            x=('x','first'),y=('y','first'),w=('w','first'),h=('h','first')))

for page,page_components in union.groupby('page',sort=False):
    g=full[full.page==page].reset_index(drop=True)
    observed=load_gray(files[page]['observed']); a1=load_gray(files[page]['a1'])
    labels=compmaps[page]; gy=g.y.to_numpy(int); gx=g.x.to_numpy(int); truth=g.truth.to_numpy(int)
    for component in page_components.itertuples(index=False):
        cid=int(component.component_id); x0,y0,x1,y1=cropbox(observed.shape,int(component.x),int(component.y),int(component.w),int(component.h))
        oc=observed[y0:y1,x0:x1].copy(); a1c=a1[y0:y1,x0:x1].copy(); region=(labels[y0:y1,x0:x1]==cid)
        inside=(gy>=y0)&(gy<y1)&(gx>=x0)&(gx<x1)
        truth_map=np.full(oc.shape,-1,np.int8); truth_map[gy[inside]-y0,gx[inside]-x0]=truth[inside]
        images=[oc,a1c]; titles=['Observed','A1 baseline']; metrics=[]
        for threshold in THRESHOLDS:
            selected=g[f'sel_{threshold:.3f}'].to_numpy(bool)&inside
            sm=np.zeros(oc.shape,bool); sm[gy[selected]-y0,gx[selected]-x0]=True; sm &= region
            good=sm&(truth_map==TRUTH_PRINT); bad=sm&(truth_map==TRUTH_ANNOTATION); unc=sm&(truth_map==TRUTH_UNCERTAIN)
            candidate=a1c.copy(); yy,xx=np.where(sm); candidate[yy,xx]=oc[yy,xx]
            view=overlay(oc,[(dilate(good),GREEN),(dilate(bad),RED),(dilate(unc),PURPLE)])
            diff=np.clip(cv2.absdiff(a1c,candidate).astype(np.int16)*4,0,255).astype(np.uint8)
            images.extend([candidate,view,diff]); titles.extend([f'A5 {threshold:.3f}',f'Truth {threshold:.3f}',f'|A1-A5| x4 {threshold:.3f}'])
            metrics.append(f'{threshold:.3f}: {int(sm.sum())} selected, {int(good.sum())} print, {int(bad.sum())} annotation, {int(unc.sum())} uncertain')
        saved=save_component_sheet(images,titles,CROPS/page/f'{page}_component_{cid:04d}.jpg',f'{page} component {cid} | '+' | '.join(metrics))
        rows.append({'page':page,'source_group':component.source_group,'component_id':cid,'crop_path':str(saved),
                     'max_print_restored':int(component.max_print_restored),'max_annotation_retained':int(component.max_annotation_retained),
                     'max_uncertain':int(component.max_uncertain),'ever_catastrophic':bool(component.ever_catastrophic)})
    gc.collect()

crop_index=pd.DataFrame(rows)
crop_index.to_csv(TABLES/'component_crop_index.csv',index=False)
print('Component sheets:',len(crop_index))


## 5. Review queues, summary and upload package


In [ ]:
queues = {
    'catastrophic': crop_index[crop_index.ever_catastrophic].sort_values(['max_annotation_retained','max_print_restored'],ascending=[False,False]),
    'annotation_heavy': crop_index.sort_values(['max_annotation_retained','max_print_restored'],ascending=[False,False]),
    'beneficial': crop_index[crop_index.max_print_restored>0].sort_values(['max_annotation_retained','max_print_restored'],ascending=[True,False]),
}
for name,q in queues.items():
    q.head(MAX_QUEUE).to_csv(TABLES/f'{name}_review_queue.csv',index=False)

plt.figure(figsize=(9,6))
plt.plot(summary.annotation_retained_per_print_restored,summary.coverage,marker='o')
for _,r in summary.iterrows():
    plt.annotate(f'{r.threshold:.3f}',(r.annotation_retained_per_print_restored,r.coverage))
plt.xlabel('Annotation retained per genuine print pixel restored')
plt.ylabel('Eligible PRINT coverage')
plt.title('A5 safety / coverage frontier')
plt.tight_layout(); plt.savefig(VISUAL/'safety_coverage_frontier.png',dpi=180); plt.close()

env={'python':sys.version,'opencv':cv2.__version__,'numpy':np.__version__,'pandas':pd.__version__,'platform':platform.platform()}
(OUT/'software_environment.json').write_text(json.dumps(env,indent=2))
identity={'created_utc':datetime.now(timezone.utc).isoformat(),'version':'notebook89r-a5-candidate-audit',
          'development_only':True,'promotion_authority':False,'retrained':False,'refitted':False,'recalibrated':False,
          'source_run':str(SOURCE),'source_predictions_sha256':sha256_file(PRED),'thresholds':THRESHOLDS,
          'truth_semantics':{'0':'ANNOTATION','1':'PRINT','2':'UNCERTAIN'},
          'probability_semantics':'PRINT-oriented, empirically checked','candidate_contract':'A1 plus exact observed bytes at selected PRINT coordinates',
          'pages':int(full.page.nunique()),'rows':int(len(full)),'component_crops':int(len(crop_index)),'software':env}
(OUT/'experiment_identity.json').write_text(json.dumps(identity,indent=2))

text='NOTEBOOK 89R A5 CANDIDATE AUDIT

Source: '+str(SOURCE)+'
Retrained: False
Refitted: False
Recalibrated: False

'+summary.to_string(index=False)+'

Green=print restored; red=annotation retained; purple=truth uncertain.
'
(OUT/'executive_summary.txt').write_text(text)

curated=[OUT/'executive_summary.txt',OUT/'experiment_identity.json',OUT/'software_environment.json',
         AUDIT/'probability_directionality.csv',AUDIT/'schema_resolution.json',
         TABLES/'threshold_summary.csv',TABLES/'results_by_page.csv',TABLES/'results_by_component.csv',
         TABLES/'component_crop_index.csv',VISUAL/'safety_coverage_frontier.png'] + [TABLES/f'{name}_review_queue.csv' for name in queues]
for p in curated:
    require(p.exists(),f'Missing required upload artefact: {p}')
    dst=UPLOAD/p.relative_to(OUT); dst.parent.mkdir(parents=True,exist_ok=True); shutil.copy2(p,dst)
(UPLOAD/'README.txt').write_text('Notebook 89R review bundle. A5 candidate = A1 plus exact observed bytes at selected PRINT coordinates. Green=print restored; red=annotation retained; purple=uncertain.
')
manifest=[]
for p in sorted(UPLOAD.rglob('*')):
    if p.is_file(): manifest.append({'relative_path':str(p.relative_to(UPLOAD)),'bytes':p.stat().st_size,'sha256':sha256_file(p)})
pd.DataFrame(manifest).to_csv(UPLOAD/'upload_manifest.csv',index=False)
zip_path=UPLOAD/'notebook89r_upload_bundle.zip'
with zipfile.ZipFile(zip_path,'w',zipfile.ZIP_DEFLATED) as z:
    for p in sorted(UPLOAD.rglob('*')):
        if p.is_file() and p!=zip_path: z.write(p,p.relative_to(UPLOAD))
print(text)
print('Upload bundle:',zip_path)


## Inspect first

- `audit/probability_directionality.csv`
- `tables/threshold_summary.csv`
- `tables/catastrophic_review_queue.csv`
- `tables/annotation_heavy_review_queue.csv`
- `tables/beneficial_review_queue.csv`
- `crops/<page>/<page>_component_<id>.jpg`
- `whole_pages/<page>/a5_<threshold>.png`
- `upload/notebook89r_upload_bundle.zip`
